# 🏢 Airpeace Knowledge Worker

A RAG-based knowledge worker for Airpeace Limited - Nigeria's largest indigenous airline.

This notebook combines:
- **Ingestion**: Loading documents, creating chunks, and building embeddings
- **Answer**: RAG-based question answering with context retrieval
- **App**: Gradio interface for interactive chat

## 1. Setup and Imports

In [ ]:
import os
import glob
from pathlib import Path

import gradio as gr
from dotenv import load_dotenv

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import SystemMessage, HumanMessage, convert_to_messages
from langchain_core.documents import Document

load_dotenv(override=True)

## 2. Configuration

In [2]:
MODEL = "gpt-4.1-nano"
RETRIEVAL_K = 10

DB_NAME = "vector_db"
KNOWLEDGE_BASE = "knowledge-base"

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Airpeace Limited.
You are chatting with a user about Airpeace Limited.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

## 3. Ingestion Functions

These functions handle loading documents from the knowledge base, splitting them into chunks, and creating the vector embeddings.

In [3]:
def fetch_documents():
    """Load all markdown documents from the knowledge base folders."""
    folders = glob.glob(str(Path(KNOWLEDGE_BASE) / "*"))
    documents = []
    for folder in folders:
        doc_type = os.path.basename(folder)
        loader = DirectoryLoader(
            folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"}
        )
        folder_docs = loader.load()
        for doc in folder_docs:
            doc.metadata["doc_type"] = doc_type
            documents.append(doc)
    return documents


def create_chunks(documents):
    """Split documents into smaller chunks for better retrieval."""
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=200)
    chunks = text_splitter.split_documents(documents)
    return chunks


def create_vectorstore(chunks):
    """Create or replace the vector store with document embeddings."""
    if os.path.exists(DB_NAME):
        Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()

    vectorstore = Chroma.from_documents(
        documents=chunks, embedding=embeddings, persist_directory=DB_NAME
    )

    collection = vectorstore._collection
    count = collection.count()

    sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
    dimensions = len(sample_embedding)
    print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")
    return vectorstore

## 4. Run Ingestion

Execute this cell to load documents and create the vector database.

In [ ]:
documents = fetch_documents()
print(f"Loaded {len(documents)} documents")

chunks = create_chunks(documents)
print(f"Created {len(chunks)} chunks")

vectorstore = create_vectorstore(chunks)
print("Ingestion complete!")

## 5. RAG Answer Functions

These functions handle retrieving relevant context and generating answers using the LLM.

In [5]:
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)


def fetch_context(question: str) -> list[Document]:
    """Retrieve relevant context documents for a question."""
    return retriever.invoke(question, k=RETRIEVAL_K)


def combined_question(question: str, history: list[dict] = []) -> str:
    """Combine all the user's messages into a single string for better context retrieval."""
    prior = "\n".join(m["content"] for m in history if m["role"] == "user")
    return prior + "\n" + question


def answer_question(question: str, history: list[dict] = []) -> tuple[str, list[Document]]:
    """Answer the given question with RAG; return the answer and the context documents."""
    combined = combined_question(question, history)
    docs = fetch_context(combined)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    messages = [SystemMessage(content=system_prompt)]
    messages.extend(convert_to_messages(history))
    messages.append(HumanMessage(content=question))
    response = llm.invoke(messages)
    return response.content, docs

## 6. Test the RAG System

Test the question answering before launching the full UI.

In [ ]:
test_question = "What is Airpeace and when was it founded?"
answer, context_docs = answer_question(test_question)

print("Question:", test_question)
print("\nAnswer:", answer)
print(f"\nRetrieved {len(context_docs)} context documents")

## 7. Gradio Chat Interface

The interactive chat application for the Airpeace Knowledge Worker.

In [7]:
def format_context(context):
    """Format the retrieved context for display in the UI."""
    result = "<h2 style='color: #ff7800;'>Relevant Context</h2>\n\n"
    for doc in context:
        result += f"<span style='color: #ff7800;'>Source: {doc.metadata['source']}</span>\n\n"
        result += doc.page_content + "\n\n"
    return result


def chat(history):
    """Process chat messages and generate responses."""
    last_message = history[-1]["content"]
    prior = history[:-1]
    answer, context = answer_question(last_message, prior)
    history.append({"role": "assistant", "content": answer})
    return history, format_context(context)


def put_message_in_chatbot(message, history):
    """Add user message to chat history."""
    return "", history + [{"role": "user", "content": message}]

## 8. Launch the Application

Run this cell to launch the Gradio chat interface.

In [ ]:
theme = gr.themes.Soft(font=["Inter", "system-ui", "sans-serif"])

with gr.Blocks(title="Airpeace Knowledge Worker", theme=theme) as ui:
    gr.Markdown("# 🏢 Airpeace Knowledge Worker\nAsk me anything about Airpeace!")

    with gr.Row():
        with gr.Column(scale=1):
            chatbot = gr.Chatbot(
                label="💬 Conversation", height=600, type="messages", show_copy_button=True
            )
            message = gr.Textbox(
                label="Your Question",
                placeholder="Ask anything about Airpeace...",
                show_label=False,
            )

        with gr.Column(scale=1):
            context_markdown = gr.Markdown(
                label="📚 Retrieved Context",
                value="*Retrieved context will appear here*",
                container=True,
                height=600,
            )

    message.submit(
        put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]
    ).then(chat, inputs=chatbot, outputs=[chatbot, context_markdown])

ui.launch(inbrowser=True)